# FastConformer fine-tune - bản đang làm

Notebook này là bản sandbox cá nhân để thử cấu hình FastConformer trên VIVOS trước khi chốt run cuối. Phần train/eval/deploy nằm trong package `src/asr_lab`; notebook chỉ giữ cấu hình, lệnh chạy Kaggle, đọc kết quả và ghi chú thí nghiệm.


## Hướng fine-tune hiện tại

- Model nền: `nvidia/stt_en_fastconformer_transducer_large`.
- Dữ liệu chính: VIVOS, text được normalize tiếng Việt trước khi train/eval.
- Cách train: thay tokenizer/vocabulary tiếng Việt, rồi fine-tune full encoder + decoder RNNT qua module main `asr_lab.train.finetune_vivos`.
- Mặc định notebook để `RUN_REMOTE = False` để xem lệnh trước; đổi sang `True` khi muốn chạy Kaggle thật.


In [ ]:
from pathlib import Path
import sys


def find_local_root(start=None):
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "asr_lab").is_dir():
            return candidate
    raise RuntimeError("Không tìm thấy ASR_local/src/asr_lab")


LOCAL_ROOT = find_local_root()
LOCAL_SRC = LOCAL_ROOT / "src"
if str(LOCAL_SRC) not in sys.path:
    sys.path.insert(0, str(LOCAL_SRC))

from asr_lab.paths import bootstrap

LOCAL_ROOT, MAIN_ROOT = bootstrap(LOCAL_ROOT)
print("LOCAL_ROOT =", LOCAL_ROOT)
print("MAIN_ROOT  =", MAIN_ROOT)


In [ ]:
from dataclasses import asdict

from IPython.display import display

from asr_lab.analytics.local_compare import (
    artifact_manifest,
    load_results,
    result_table,
    run_main_report_commands,
)
from asr_lab.data.vivos import prepare_vivos_manifests
from asr_lab.deploy.local_kaggle import (
    build_code_dataset,
    poll_kernel,
    pull_kernel,
    push_common_voice_resume,
    push_vivos_finetune,
    run_cmd,
    upload_resume_dataset,
    uv_module,
)
from asr_lab.model.fastconformer import CommonVoiceRunConfig, FastConformerRunConfig


## Cấu hình thí nghiệm đang làm


In [ ]:
RUN_REMOTE = False
ACCOUNT = "trnmnhtn"
WORK_TAG = "work"

config = FastConformerRunConfig(
    account=ACCOUNT,
    kernel=f"asr-ft-fc115m-{WORK_TAG}",
    run_id=f"vivos-fc115m-{WORK_TAG}",
    model="nvidia/stt_en_fastconformer_transducer_large",
    epochs=50,
    batch=16,
    vocab_size=1024,
    lr="2e-4",
    precision="32",
    max_minutes=480,
)

display(asdict(config))
print("script args:")
print(config.script_args())


## Kiểm tra account Kaggle


In [ ]:
run_cmd(
    uv_module("asr_lab.deploy.kaggle", "accounts"),
    MAIN_ROOT,
    RUN_REMOTE,
)


## Chuẩn bị manifest VIVOS local nếu cần soi nhanh


In [ ]:
PREPARE_LOCAL_MANIFESTS = False

if PREPARE_LOCAL_MANIFESTS:
    manifests = prepare_vivos_manifests(MAIN_ROOT / "data" / "vivos", val_n=300)
    display(manifests.as_dict())
    print("rows:", manifests.train_rows, manifests.val_rows, manifests.test_rows)
else:
    print("Bỏ qua manifest local; script Kaggle sẽ tự chuẩn bị dữ liệu khi chạy train.")


## Build code dataset cho Kaggle


In [ ]:
build_code_dataset(config, MAIN_ROOT, RUN_REMOTE)


## Push kernel fine-tune VIVOS


In [ ]:
push_vivos_finetune(config, MAIN_ROOT, RUN_REMOTE)


## Theo dõi và tải artifact


In [ ]:
poll_kernel(config.account, config.kernel, MAIN_ROOT, RUN_REMOTE)
pull_kernel(config.account, config.kernel, MAIN_ROOT, RUN_REMOTE)


## Đọc kết quả WER


In [ ]:
results = load_results(MAIN_ROOT, config.run_id)
display(result_table(results))


## So sánh với baseline khi đã có results.json


In [ ]:
BASE_RUN_ID = "vivos-fc115m-v1"
RUN_MAIN_REPORTS = False

if RUN_MAIN_REPORTS:
    run_main_report_commands(MAIN_ROOT, BASE_RUN_ID, config.run_id)
else:
    print("Đặt RUN_MAIN_REPORTS=True sau khi đã pull artifact/results.json.")


## Artifact của run đang làm


In [ ]:
display(artifact_manifest(MAIN_ROOT, config.run_id))


## Ghi chú thí nghiệm

- Nếu Kaggle hết thời gian, giữ nguyên `run_id` riêng cho bản đang làm để không đè run final.
- Nếu WER xấu, kiểm tra lại normalization, vocabulary size và log train/val loss trước khi đổi model.
- Khi run đã ổn, chuyển cấu hình sang notebook `02_fastconformer_main.ipynb` để chạy bản cuối theo main.
